### 📝 **Enunciado**

O significado de uma frase pode ser totalmente alterado pela simples presença de um advérbo de negação, como **"Não"**, **"Nunca"**, etc. Com base nisso, utilize o arquivo desenvolvido e aplique uma lógica adicional para esse caso.

### 01 - Instalação (Bibliotecas)

In [ ]:
!pip install sounddevice

!pip install scipy

!pip install openai-whisper

### 02 - Recursos (Importações)

In [ ]:
import sounddevice as sd
import scipy.io.wavfile as wav
import whisper

from whisper import Whisper
from datetime import datetime

### 03 - Funções

In [ ]:
def record_sound(file_path: str, record_duration: int = 5, sample_rate = 44100):
  print("\nGravando Áudio...")

  sound = sd.rec(record_duration * sample_rate, samplerate=sample_rate, channels=1)

  sd.wait()

  wav.write(file_path, sample_rate, sound)

  print("\nGravação Finalizada.")

def transcribe_sound(model: Whisper, file_path: str):
  print("\nTranscrevendo Áudio...")

  result = model.transcribe(file_path, language="pt")

  text = result["text"]

  print("\nTranscrição Concluída.")

  return text.lower()

def verify_denial(text: str, target_tokens: list[str]):
  denial_tokens = ["nao", "não", "nunca", "tampouco", "nem", "jamais"]

  normalized_text = text.lower()

  return all(token in normalized_text for token in target_tokens) and not any(denial in normalized_text for denial in denial_tokens)

def classify_commands(text: str):
  if (verify_denial(text, ["luz", "ligar"])):
    print("\nLigue a luz.")
  elif (verify_denial(text, ["luz", "desligar"])):
    print("\nDesligue a luz.")
  elif (verify_denial(text, ["hora"])):
    print("\nHora: ", datetime.now().strftime("%H:%M"))
  elif (verify_denial(text, ["data"])):
    print("\nData: ", datetime.now().strftime("%d/%m/%Y"))
  else:
    print("\nComando não reconhecido.")

def execute_pipeline(model: Whisper, file_path: str):
  record_sound(file_path)

  text = transcribe_sound(model, file_path)

  classify_commands(text)

  global sound_text

  sound_text = text

### 04 Modelo Whisper

In [ ]:
print("Baixando Modelo...\n")

model = whisper.load_model("medium")

print("\nModelo Carregado.")

### 05 - Execução (Pipeline)

In [ ]:
file_path = "./sound.wav"

execute_pipeline(model, file_path)

print(f"\nTexto: {sound_text}")